In [1]:
# M5.2 -- Stacking & Unstacking 
# both .stack() and .unstack() work with multi-level indexes
# in short, they move index levels to columns and columns to index levels

# .stack() take a column level and moves it to the row-index level
# this makes the dataframe long or tabular

# .unstack() takea a row index level and moves it to the column level
# this makes the dataframe shorter and wider

# previously, .melt() and .pivot() were used to achieve similar results
# the difference is that .stack() and .unstack() operate on the index not named columns
# so if you have a multi-level index by "datetime" and "sku_id" you should operate on the dataframe with .stack() and .unstack()

# import libraries
import pandas as pd
import numpy as np

# quick notes: tuples
# a tuple is a base python data structure -- an ordered immutable collection of values wrapped in parentheses
# immutable signifies that once it is created it can no longer be changed
# in practice -- tuples are often used as dictionary keys and/or as labels for multi-level indexes because of their immutability
# example: my_list = [1, 2, and 3] versus my_tuple = (1,2, 3)
# example: my_list[0] = 99 versus my_tuple[0] = 99 <-- my_tuple[0] returns an error becuse you cant modify a tuple


# create a multi-index dataframe -- using tuples
# pass a dictionary to the pd.DataFrame() 
# use two-level column names ("East", "units")
# using a two-level column name the first element ("East") is the top-level and the second element ("units") is the metric

# what do the .from_tuples() do?
# tuples are a pair or group of values that belong together and do not change
# pandas reads the dictionary keys as tuples
# without .from_tuples(df.columns, names=["warehouse", "metrics"]) each of the multiple levels would be unnamed
# instead you'd have to reference them by number (level 0) and (level 1)
# by naming them you can .stack() and .unstack() them

df = pd.DataFrame(
    {
        ("East", "units"): [120, 300, 88],
        ("East", "cost"): [25.50, 15.75, 8.25],
        ("West", "units"): [45, 0, 210],
        ("West", "cost"): [42.00, 15.75, 8.25]
    },
    index=["SKU-001", "SKU-002", "SKU-003"]
)

df.columns = pd.MultiIndex.from_tuples(df.columns, names=["warehouse", "metrics"])
df.index.name = "sku_id"
print(df)


warehouse  East         West       
metrics   units   cost units   cost
sku_id                             
SKU-001     120  25.50    45  42.00
SKU-002     300  15.75     0  15.75
SKU-003      88   8.25   210   8.25


In [4]:
# exercise 2

# .stack() the warehouse level into the index
# then filter the rows for units > 50

df_stack = (
    df
    .stack(level="warehouse")
)
print(df_stack[df_stack["units"] > 50])

metrics            units   cost
sku_id  warehouse              
SKU-001 East         120  25.50
SKU-002 East         300  15.75
SKU-003 East          88   8.25
        West         210   8.25


In [ ]:
# exercise 3

# use .unstack()
# unstack the warehouse level back to columns -- making the dataframe shorter and wider

df_unstack = (
    df_stack
    .unstack(level="warehouse")
)

print(df_unstack)

# confirm it matches the original df dataframe
df = df_unstack

# f-string
print(f"Dataframes match: {df.equals(df_unstack)}")



metrics   units        cost       
warehouse  East West   East   West
sku_id                            
SKU-001     120   45  25.50  42.00
SKU-002     300    0  15.75  15.75
SKU-003      88  210   8.25   8.25
Dataframes match: True


In [ ]:
# exercise 4 -- combine groupby + unstack
# rebuild df_long

# rebuild df_long from M5.1
df_wide = pd.DataFrame({
    "sku_id":   ["SKU-001", "SKU-002", "SKU-003"],
    "category": ["Widget", "Gadget", "Component"],
    "Jan": [100, 60, 200], "Feb": [85, 75, 180],
    "Mar": [120, 90, 220], "Apr": [95, 80, 195]
})

df_long = df_wide.melt(
    id_vars=["sku_id", "category"],
    value_vars=["Jan", "Feb", "Mar", "Apr"],
    var_name="month",
    value_name="demand"
)

# use groupby + unstack
group_df = (
    df_long
    .groupby(["category", "month"])["demand"]
    .sum()
    .unstack("month")
)
print(group_df)

# the months appear out of order in group_df because they are not in datetime format
# the months appear as strings
# put them in order

month_order = ["Jan", "Feb", "Mar", "Apr"]
group_df = (
    df_long
    .assign(
        month = lambda x: pd.Categorical(x["month"], categories=month_order, ordered=True)
    )
    .groupby(["category", "month"])["demand"]
    .sum()
    .unstack("month")
)

# filter the data for rows where any month > 200
print(group_df[group_df.gt(200).any(axis=1)])

# what does unstack do to a groupby result?
# it converts the group series multi level indexes into a readable cross-tab style dataframe
# in this example, the categories become rows, months become columns
# the result is an organized time-series matrix that could easily be plotted

# what 

month      Apr  Feb  Jan  Mar
category                     
Component  195  180  200  220
Gadget      80   75   60   90
Widget      95   85  100  120
month      Jan  Feb  Mar  Apr
category                     
Component  200  180  220  195
